# 09 -- Real-Terrain Dynamic Trajectory Planning

Edit the input cell below to point at your georeferenced DEM and precomputed
sunlight archive, then compare GridRunner and safe-interval paths visually.
This notebook follows `examples/trajectory_real_terrain.py`.

The `.npz` archive must contain exactly `sunlight` (`uint8`, shaped
`(interval, y, x)`) and `boundaries_utc` (`interval + 1` ISO-8601 UTC strings).
No horizons are generated and no GPU is required.

## Setup and editable inputs

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError("Cannot locate Lunarscout repository root.")

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))


import hashlib
import json
import time
import tracemalloc
from datetime import datetime, timezone

import lunarscout as ls
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

# EDIT THESE VALUES.
DEM_PATH = None                 # Path("/data/site/dem.tif")
SUNLIGHT_ARCHIVE = None        # Path("/data/site/sunlight.npz")
START = None                    # (x, y), for example (120, 80)
GOAL = None                     # (x, y), for example (400, 320)
MINIMUM_SUNLIGHT = 0.2
SPEED_M_PER_HOUR = 36.0
REPORT_PATH = None              # Optional: Path("trajectory-report.json")

READY = all(value is not None for value in
            (DEM_PATH, SUNLIGHT_ARCHIVE, START, GOAL))
print("Inputs ready." if READY else "Edit DEM_PATH, SUNLIGHT_ARCHIVE, START, and GOAL, then rerun.")

## Load and validate the DEM and sunlight archive

In [ ]:
if READY:
    DEM_PATH = Path(DEM_PATH).expanduser().resolve()
    SUNLIGHT_ARCHIVE = Path(SUNLIGHT_ARCHIVE).expanduser().resolve()
    elevation_m, georef = ls.read_geotiff(DEM_PATH)
    if georef is None:
        raise ValueError("DEM must contain complete georeferencing")
    with np.load(SUNLIGHT_ARCHIVE, allow_pickle=False) as archive:
        if set(archive.files) != {"boundaries_utc", "sunlight"}:
            raise ValueError("Archive must contain only boundaries_utc and sunlight")
        sunlight = np.array(archive["sunlight"], copy=True)
        boundary_text = [str(value) for value in archive["boundaries_utc"]]
    boundaries = tuple(datetime.fromisoformat(
        value[:-1] + "+00:00" if value.endswith(("Z", "z")) else value
    ).astimezone(timezone.utc) for value in boundary_text)
    valid = np.isfinite(elevation_m)
    if georef.nodata is not None:
        valid &= elevation_m != georef.nodata
    print(f"Grid: {georef.width} x {georef.height}")
    print(f"Intervals: {len(boundaries) - 1}, {boundaries[0]} to {boundaries[-1]}")

## Inspect terrain and configuration-space samples

In [ ]:
if READY:
    sample_indices = sorted(set((0, len(sunlight)//2, len(sunlight)-1)))
    fig, axes = plt.subplots(1, len(sample_indices) + 1,
                             figsize=(5 * (len(sample_indices) + 1), 5),
                             constrained_layout=True)
    terrain = np.where(valid, elevation_m, np.nan)
    image = axes[0].imshow(terrain, cmap="terrain", origin="upper")
    axes[0].set_title("DEM and endpoints")
    axes[0].scatter(*START, color="cyan", edgecolor="black", s=70)
    axes[0].scatter(*GOAL, color="red", edgecolor="white", marker="*", s=120)
    fig.colorbar(image, ax=axes[0], label="elevation (m)")
    for ax, index in zip(axes[1:], sample_indices):
        configuration_frame = valid & (sunlight[index] >= round(MINIMUM_SUNLIGHT * 255))
        ax.imshow(configuration_frame, cmap="gray", vmin=0, vmax=1, origin="upper")
        ax.set_title(boundaries[index].isoformat())
    for ax in axes:
        ax.set(xlabel="x (cell)", ylabel="y (cell)")
    plt.show()

## Run both exact CPU algorithms

In [ ]:
if READY:
    signal = ls.trajectory.ArraySunlightProvider(boundaries, sunlight, georef)
    configuration = ls.trajectory.AllOfConfigurationSpaceProvider((
        ls.trajectory.StaticConfigurationSpaceProvider(valid, georef),
        ls.trajectory.SunlightThresholdProvider(signal, MINIMUM_SUNLIGHT),
    ))
    model = ls.trajectory.StaticTravelModel(
        speed_m_per_h=SPEED_M_PER_HOUR, include_diagonals=True
    )
    results, measurements = {}, {}
    for algorithm in ("gridrunner", "safe_interval"):
        tracemalloc.start()
        started = time.perf_counter()
        results[algorithm] = ls.trajectory.dynamic_path(
            valid, georef, START, GOAL, boundaries, configuration, boundaries[0],
            valid=valid, elevation=elevation_m, model=model,
            algorithm=algorithm, backend="cpu",
        )
        elapsed = time.perf_counter() - started
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        measurements[algorithm] = {"runtime_seconds": elapsed,
                                   "peak_tracemalloc_bytes": peak}
        result = results[algorithm]
        print(f"{algorithm:13s}: reachable={result.reachable}, "
              f"travel={result.travel_time_hours}, runtime={elapsed:.3f}s")

## Overlay paths and arrival times

In [ ]:
if READY:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    for ax, (algorithm, result) in zip(axes, results.items()):
        ax.imshow(np.where(valid, elevation_m, np.nan), cmap="terrain", origin="upper")
        if result.reachable:
            hours = np.array([(value - boundaries[0]).total_seconds() / 3600
                              for value in result.arrival_times])
            ax.plot(result.cells[:, 0], result.cells[:, 1], color="white", lw=2)
            points = ax.scatter(result.cells[:, 0], result.cells[:, 1], c=hours,
                                cmap="plasma", s=18, edgecolor="black", linewidth=0.2)
            fig.colorbar(points, ax=ax, label="hours after departure")
        ax.set(title=algorithm, xlabel="x (cell)", ylabel="y (cell)")
    plt.show()

## Build an editable reproducibility report

In [ ]:
if READY:
    def sha256(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as stream:
            while chunk := stream.read(1024 * 1024):
                digest.update(chunk)
        return digest.hexdigest()

    report = {
        "dem": {"path": str(DEM_PATH), "sha256": sha256(DEM_PATH)},
        "sunlight": {"path": str(SUNLIGHT_ARCHIVE),
                     "sha256": sha256(SUNLIGHT_ARCHIVE)},
        "grid": [georef.width, georef.height],
        "start": list(START), "goal": list(GOAL),
        "environment_sampling": {
            "boundaries_utc": [value.isoformat() for value in boundaries],
            "minimum_sunlight": MINIMUM_SUNLIGHT,
        },
        "model": {"speed_m_per_h": SPEED_M_PER_HOUR,
                  "include_diagonals": True, "slip": None},
        "results": {
            name: {**measurements[name], "reachable": result.reachable,
                   "arrival_time": None if result.arrival_time is None else result.arrival_time.isoformat(),
                   "travel_time_hours": result.travel_time_hours,
                   "path_cell_count": None if result.cells is None else len(result.cells)}
            for name, result in results.items()
        },
    }
    print(json.dumps(report, indent=2))
    if REPORT_PATH is not None:
        Path(REPORT_PATH).write_text(json.dumps(report, indent=2) + "\n")
        print(f"Wrote {REPORT_PATH}")